![Redis](https://redis.io/wp-content/uploads/2024/04/Logotype.svg?auto=webp&quality=85,75&width=120)

# Redis Cloud Agent Memory with the NVIDIA NeMo Agent Toolkit

## Introduction

This is the managed-cloud twin of `06_nemo_agent_toolkit_redis.ipynb`. Same agent, same [**nemo-agent-toolkit-redis**](https://github.com/redis-developer/nemo-agent-toolkit-redis) wiring — but instead of running the open-source [Agent Memory Server](https://github.com/redis/agent-memory-server) ourselves, we point the [**NVIDIA NeMo Agent Toolkit**](https://github.com/NVIDIA/NeMo-Agent-Toolkit) at **[Redis Cloud Agent Memory](https://redis.io/agent-memory/)**, a fully managed service.

No Docker, no server to operate, no worker to babysit — just an endpoint and an API key.

## What changes vs. the self-hosted recipe

Only the deployment and connection details:

| | Self-hosted (nb 06) | Redis Cloud (this nb) |
|---|---|---|
| Server | you run it via Docker | fully managed |
| `base_url` | `http://localhost:8000` | your Cloud service endpoint |
| Auth | disabled (dev) | API key as `Authorization: Bearer` |
| Ops | you scale/patch it | Redis handles it |

The agent code and workflow config are otherwise identical.

## Let's Begin
<a href="https://colab.research.google.com/github/redis-developer/redis-ai-resources/blob/main/python-recipes/agents/07_nemo_agent_toolkit_redis_cloud.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Prerequisites

1. [Create a database on Redis Cloud](https://redis.io/docs/latest/operate/rc/databases/create-database) (a [free account](https://redis.io/try-free/) works).
2. [Create an Agent Memory service](https://redis.io/docs/latest/operate/rc/context-engine/agent-memory/create-service) for that database.
3. From the service's **Configuration** page, grab the **API endpoint**, the **Store ID**, and the **API key** (shown only once at creation — [regenerate](https://redis.io/docs/latest/operate/rc/context-engine/agent-memory/view-service#replace-service-api-key) if lost).
4. An **OpenAI API key** for the chat LLM.

In [ ]:
# NBVAL_SKIP
%pip install -q nemo-agent-toolkit-redis requests

## Set environment variables

Point the toolkit at your managed endpoint and supply the API key.

In [ ]:
# NBVAL_SKIP
import os, getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

# From your Agent Memory service Configuration page on Redis Cloud:
#   Settings > General settings  ->  API endpoint + Store ID
#   (the API key is shown only once, when you create/replace the service key)
os.environ["REDIS_AGENT_MEMORY_URL"] = input("Agent Memory API endpoint (e.g. https://<service>.agent-memory.redis.cloud): ").strip()
os.environ["REDIS_AGENT_MEMORY_API_KEY"] = getpass.getpass("Agent Memory API key: ")
os.environ["REDIS_AGENT_MEMORY_NAMESPACE"] = "nat-auto-memory"
os.environ["NAT_OPENAI_MODEL"] = "gpt-4o-mini"

# Managed Cloud requires the API key as a Bearer token on every request.
AUTH_HEADERS = {"Authorization": f"Bearer {os.environ['REDIS_AGENT_MEMORY_API_KEY']}"}

## Define the NAT workflow

Identical to the self-hosted config except the memory backend now targets the Cloud endpoint and passes the API key. The `api_key` value is sent as a Bearer token on every request to the managed service.

> If your installed toolkit version names the auth field differently (e.g. `token`), adjust the `redis_agent_memory_backend` block accordingly — run `nat info components` to see its schema.

In [ ]:
config_yaml = """general:
  telemetry:
    enabled: false

llms:
  openai_llm:
    _type: openai
    model_name: ${NAT_OPENAI_MODEL:-gpt-4o-mini}

functions:
  assistant_chat:
    _type: chat_completion
    llm_name: openai_llm
    system_prompt: >-
      You are a helpful assistant. When memory context is provided, use it to
      answer consistently about the user's preferences and prior facts.

memory:
  redis_ltm:
    _type: redis_agent_memory_backend
    base_url: ${REDIS_AGENT_MEMORY_URL}
    api_key: ${REDIS_AGENT_MEMORY_API_KEY}
    default_namespace: ${REDIS_AGENT_MEMORY_NAMESPACE:-nat-auto-memory}

workflow:
  _type: redis_agent_memory_auto_memory
  description: >-
    A chat agent that uses Redis Agent Memory working memory plus memory_prompt
    hydration on every turn.
  inner_agent_name: assistant_chat
  memory_name: redis_ltm
  default_user_id: demo-user
  default_session_id: demo-session
  memory_prompt:
    optimize_query: false
    long_term_search:
      limit: 5
  working_memory:
    namespace: ${REDIS_AGENT_MEMORY_NAMESPACE:-nat-auto-memory}
    model_name: ${NAT_OPENAI_MODEL:-gpt-4o-mini}
    ttl_seconds: 86400
    long_term_memory_strategy:
      strategy: discrete
"""

with open("nat_config.yml", "w") as f:
    f.write(config_yaml)
print("wrote nat_config.yml")

## Run the agent

Exactly the same driver as the self-hosted recipe — the code doesn't know or care that memory now lives in Redis Cloud.

In [ ]:
# NBVAL_SKIP
import asyncio
from pathlib import Path

from nat.utils import run_workflow

CONFIG_FILE = Path("nat_config.yml").resolve()


async def chat(prompt: str, user_id: str = "demo-user", conversation_id: str = "demo-session") -> str:
    """Run one turn through the NAT auto-memory workflow.

    NAT maps user_id -> Redis Agent Memory user_id and conversation_id -> session_id,
    so working memory is hydrated and turns are captured automatically.
    """
    result = await run_workflow(
        config_file=CONFIG_FILE,
        prompt=prompt,
        to_type=str,
        session_kwargs={"conversation_id": conversation_id, "user_id": user_id},
    )
    print(f"User: {prompt}")
    print(f"Assistant: {result}\n")
    return result

In [ ]:
# NBVAL_SKIP
# A multi-turn conversation. Turn 3 relies on memory captured in turns 1-2.
await chat("Hi! My name is Justin and my favorite city is Lisbon.")
await chat("I'm a vegetarian, by the way.")
# New turn -> the auto-memory wrapper hydrates prior facts from Redis Agent Memory
await chat("Where should I plan a food trip, and what should I keep in mind?")

The third answer again reflects the name / favorite city / diet captured in earlier turns — this time recalled from the managed Cloud service.

## Inspect long-term memory

Same REST call as before, but authenticated with the Cloud API key (`AUTH_HEADERS`).

In [ ]:
# NBVAL_SKIP
# Inspect what got promoted to long-term memory via the Agent Memory REST API.
import os, requests

base = os.environ["REDIS_AGENT_MEMORY_URL"].rstrip("/")
namespace = os.environ.get("REDIS_AGENT_MEMORY_NAMESPACE", "nat-auto-memory")

resp = requests.post(
    f"{base}/v1/long-term-memory/search",
    headers={"Content-Type": "application/json", **AUTH_HEADERS},
    json={"text": "favorite city and diet", "namespace": {"eq": namespace}, "limit": 5},
    timeout=30,
)
resp.raise_for_status()
for m in resp.json().get("memories", []):
    print(f"- [{m.get('memory_type')}] {m.get('text')}")

## Cleanup

Nothing to tear down locally. To stop incurring cost, delete or pause the Agent Memory service (and its database) from the Redis Cloud console when you're done.

## Summary

Same NeMo Agent Toolkit agent, same memory behavior — now backed by managed [Redis Cloud Agent Memory](https://redis.io/agent-memory/). Moving from the self-hosted server to production was a change of `base_url` and an API key, nothing more.